# 02 - user_logs aggregation

The problem we're solving: `user_logs_v2.csv` is 1.4GB, and the full `user_logs.csv` is ~28GB way too big for pandas to load like we did with train/members/transactions. The data is also one row per user per day, but for modeling we need one row per user (since train.csv also has one row per user).

### So the goal is: shrink ~millions of daily rows down to one summary row per msno.

plan for this notebook:
- right now it's one row per user per day - we want one row per user instead
- basically just sum/count stuff per user (total listening time, days active, etc.)
- save the result so we don't have to redo this every time we open the notebook

using `polars` for this since it's supposed to handle big files a lot better than pandas.


## Step 1: imports

same idea as last time, just with polars now.

- `pl` - short name for polars, same as `pd` for pandas
- `DATA` - where the csv files are
- `PROCESSED` - folder where we'll save the output. `mkdir(exist_ok=True)` makes the folder if it's not there yet, and doesn't error if it already is


In [7]:
import polars as pl
from pathlib import Path

DATA = Path("../data/raw")
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(exist_ok=True)


## Step 2: peek at the file

just checking what's in here before doing anything big.

- `n_rows=5` - same as `nrows` in pandas, only reads the first 5 rows
- `.schema` - shows the column types, polars' version of `.dtypes`


In [8]:
sample = pl.read_csv(DATA / "user_logs_v2.csv", n_rows=5)
print(sample)
print(sample.schema)


shape: (5, 9)
┌──────────────────────┬──────────┬────────┬────────┬───┬─────────┬─────────┬─────────┬────────────┐
│ msno                 ┆ date     ┆ num_25 ┆ num_50 ┆ … ┆ num_985 ┆ num_100 ┆ num_unq ┆ total_secs │
│ ---                  ┆ ---      ┆ ---    ┆ ---    ┆   ┆ ---     ┆ ---     ┆ ---     ┆ ---        │
│ str                  ┆ i64      ┆ i64    ┆ i64    ┆   ┆ i64     ┆ i64     ┆ i64     ┆ f64        │
╞══════════════════════╪══════════╪════════╪════════╪═══╪═════════╪═════════╪═════════╪════════════╡
│ u9E91QDTvHLq6NXjEaWv ┆ 20170331 ┆ 8      ┆ 4      ┆ … ┆ 1       ┆ 21      ┆ 18      ┆ 6309.273   │
│ 8u4QIqhrHk…          ┆          ┆        ┆        ┆   ┆         ┆         ┆         ┆            │
│ nTeWW/eOZA/UHKdD5L7D ┆ 20170330 ┆ 2      ┆ 2      ┆ … ┆ 0       ┆ 9       ┆ 11      ┆ 2390.699   │
│ EqKKFTjaAj…          ┆          ┆        ┆        ┆   ┆         ┆         ┆         ┆            │
│ 2UqkWXwZbIjs03dHLU9K ┆ 20170331 ┆ 52     ┆ 3      ┆ … ┆ 3       ┆ 84      ┆

## Step 3: scan instead of read

`read_csv` loads everything into memory right away. fine for 5 rows, not fine for 28GB.

`scan_csv` doesn't load anything yet - it just sets up a plan for what to do. nothing actually runs until `.collect()` is called later.

- `.collect_schema()` - checks the column names/types without reading the whole file


In [9]:
lazy_logs = pl.scan_csv(DATA / "user_logs_v2.csv")
print(lazy_logs.collect_schema())


Schema({'msno': String, 'date': Int64, 'num_25': Int64, 'num_50': Int64, 'num_75': Int64, 'num_985': Int64, 'num_100': Int64, 'num_unq': Int64, 'total_secs': Float64})


## Step 4: group by user

this is the main part. right now there's one row per user per day, and we want one row per user total.

- `.group_by("msno")` - buckets all the rows by user
- `.agg([...])` - for each user, calculate:
  - `pl.len()` - how many days they show up in the logs
  - `.sum()` on `num_25`, `num_50`, `num_75`, `num_985`, `num_100`, `num_unq` - add all of those up
  - `total_secs` sum and mean - total and average listening time per day
- `.collect()` - this is when it actually runs.


In [13]:
user_features = (
    lazy_logs
    .group_by("msno")
    .agg([
        pl.len().alias("num_days"),
        pl.col("num_25").sum().alias("num_25_sum"),
        pl.col("num_50").sum().alias("num_50_sum"),
        pl.col("num_75").sum().alias("num_75_sum"),
        pl.col("num_985").sum().alias("num_985_sum"),
        pl.col("num_100").sum().alias("num_100_sum"),
        pl.col("num_unq").sum().alias("num_unq_sum"),
        pl.col("total_secs").sum().alias("total_secs_sum"),
        pl.col("total_secs").mean().alias("total_secs_mean"),
    ])
    .collect()
)


## Step 5: check it worked

- `.shape` - (rows, cols). rows should be way smaller now, one per user instead of one per day
- `.head()` - first few rows, just to eyeball it and make sure it looks right


In [14]:
print(user_features.shape)
user_features.head()


(1103894, 10)


msno,num_days,num_25_sum,num_50_sum,num_75_sum,num_985_sum,num_100_sum,num_unq_sum,total_secs_sum,total_secs_mean
str,u32,i64,i64,i64,i64,i64,i64,f64,f64
"""3O/O3EvdO86r9G4pV+LRJHxkPYXmbO…",2,0,0,0,1,32,13,7454.506,3727.253
"""2EKStkolh3nysPwCd7jSPUqlk4BUoa…",15,40,11,16,34,824,863,223496.394,14899.7596
"""ta3RyRs6R4NecvI4WiqGiIXgKMtFH1…",23,5,6,1,8,283,285,76462.955,3324.476304
"""CzGOe8Mnyw1r/rw7eE5s/5d02RwD17…",30,710,133,79,97,1293,1712,367222.05,12240.735
"""Z4JEMu0iwr6LSNy6K+Qv8WS9brAiF8…",12,91,8,8,5,158,172,43764.852,3647.071


## Step 6: save as parquet

don't want to rerun all that every time we open this notebook, so save it.

- `.write_parquet(...)` - saves the table as a parquet file. smaller than csv and keeps the column types
- next time we can just use `read_parquet()` and get this table back instantly instead of redoing the aggregation


In [15]:
user_features.write_parquet(PROCESSED / "user_logs_v2_agg.parquet")
print("Saved:", PROCESSED / "user_logs_v2_agg.parquet")


Saved: ../data/processed/user_logs_v2_agg.parquet
